# 04 — Frozen Lake + Exploration
**Week 3 | RL Fundamentals**

First contact with **Gymnasium**. We run various policies on Frozen Lake and measure performance — no learning yet, just understanding the environment and the exploration-exploitation trade-off.

In [ ]:
# Install gymnasium if needed
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gymnasium', '-q'])
    import gymnasium as gym

import numpy as np
import matplotlib.pyplot as plt
print('Gymnasium version:', gym.__version__)

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=False)
print(f"Observation space: {env.observation_space}  ({env.observation_space.n} states)")
print(f"Action space:      {env.action_space}  (0=L, 1=D, 2=R, 3=U)")
print(f"\nTransition model for state 0, action 1 (DOWN):")
for prob, next_s, reward, done in env.unwrapped.P[0][1]:
    print(f"  P={prob:.2f} -> state {next_s}, reward={reward}, done={done}")

## 1. Random Policy Baseline

In [ ]:
def evaluate_policy(env, policy_fn, n_episodes=1000, max_steps=200):
    wins = 0
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        ep_return = 0
        for _ in range(max_steps):
            a = policy_fn(s)
            s, r, terminated, truncated, _ = env.step(a)
            ep_return += r
            if terminated or truncated:
                if r == 1.0: wins += 1
                break
        returns.append(ep_return)
    return wins / n_episodes, np.mean(returns)

random_win, random_ret = evaluate_policy(env, lambda s: env.action_space.sample())
print(f"Random policy:  win rate={random_win:.2%}, avg return={random_ret:.4f}")

## 2. Fixed Directional Policy
Manually specify the best deterministic path.

In [ ]:
# Hand-crafted policy for 4x4 non-slippery FrozenLake (actions: 0=L,1=D,2=R,3=U)
# Map: SFFF / FHFH / FFFH / HFFG
hand_policy = [
    1, 2, 1, 0,   # row 0
    1, 0, 1, 0,   # row 1 (H at pos 5,7)
    2, 2, 1, 0,   # row 2
    0, 2, 2, 0,   # row 3 (H at pos 12, G at 15)
]
hand_win, hand_ret = evaluate_policy(env, lambda s: hand_policy[s])
print(f"Hand policy:    win rate={hand_win:.2%}, avg return={hand_ret:.4f}")

## 3. ε-greedy Policy with Learned Bias
Start with a 'smart' action bias and add exploration noise.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
eps_values = np.linspace(0, 1, 20)
results = []
for eps in eps_values:
    def eps_hand(s, eps=eps):
        if np.random.rand() < eps:
            return env.action_space.sample()
        return hand_policy[s]
    w, _ = evaluate_policy(env, eps_hand, n_episodes=500)
    results.append(w)
ax.plot(eps_values, results, color='steelblue', linewidth=2, marker='o', markersize=5)
ax.axhline(random_win, color='tomato', linestyle='--', label=f'Random baseline ({random_win:.0%})')
ax.set_xlabel('ε (exploration rate)'); ax.set_ylabel('Win rate')
ax.set_title('Trade-off: Exploration vs Exploitation on FrozenLake')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Inspect the MDP Transitions

In [ ]:
# Show how slippery=True changes things
env_slip = gym.make('FrozenLake-v1', is_slippery=True)
slip_win, _ = evaluate_policy(env_slip, lambda s: hand_policy[s])
print(f"Hand policy on SLIPPERY lake: win rate={slip_win:.2%}")
print("\n(Our deterministic policy fails on a stochastic environment!)")

## ✅ Exercises
1. Try the 8×8 FrozenLake map (`map_name='8x8'`). Does the hand-crafted policy still work? Why not?
2. Plot win rate vs number of evaluation episodes for the random policy. How many episodes do you need for a stable estimate?
3. **Challenge**: write a systematic policy for the slippery lake — one that avoids dangerous edges. Test it.

## ✅ Exercise Solutions

### Exercise 1 — 8×8 FrozenLake

The 4×4 `hand_policy` only has 16 entries but the 8×8 map has **64 states**, so the policy immediately fails (index out of range / wrong state mapping). Even if we wrap indices, the tile layout is completely different — holes are in new positions — so the actions would send the agent straight into holes.

A simple **right-then-down** strategy works on the non-slippery 8×8 because the default map has no hole blocking the rightmost column or the bottom row.

In [ ]:
import gymnasium as gym
import numpy as np

env_8x8 = gym.make('FrozenLake-v1', map_name='8x8', is_slippery=False)
desc = env_8x8.unwrapped.desc
n = 8

print('8×8 map layout:')
for row in desc:
    print(' '.join(c.decode() for c in row))

# ── Why the 4x4 policy fails ──────────────────────────────────────────────
try:
    bad_win, _ = evaluate_policy(env_8x8, lambda s: hand_policy[s % 16], n_episodes=200)
    print(f'\n4x4 policy naively applied to 8x8: win rate = {bad_win:.2%}  (meaningless!)')
except Exception as e:
    print(f'Error: {e}')

# ── Simple right-then-down policy for 8x8 ────────────────────────────────
hand_policy_8x8 = []
for row in range(n):
    for col in range(n):
        # Go right until last column, then go down
        hand_policy_8x8.append(2 if col < n - 1 else 1)

win_8x8, _ = evaluate_policy(env_8x8, lambda s: hand_policy_8x8[s], n_episodes=1000)
print(f'\nRight-then-down policy on 8×8: win rate = {win_8x8:.2%}')
print('The 4×4 policy fails on 8×8 because: (1) wrong number of states,',
      '(2) hole positions differ — a new policy must be crafted per map.')

### Exercise 2 — How many episodes for a stable win-rate estimate?

We run the random policy `n_trials=30` times for each episode budget and track the **standard deviation** of the win-rate estimate. A stable estimate has low std — empirically this happens around **500–1000 episodes**.

In [ ]:
env_4 = gym.make('FrozenLake-v1', is_slippery=False)
np.random.seed(42)

episode_counts = [10, 50, 100, 200, 500, 1000, 2000, 5000]
n_trials = 30
means, stds = [], []

for n_ep in episode_counts:
    rates = [
        evaluate_policy(env_4, lambda s: env_4.action_space.sample(), n_episodes=n_ep)[0]
        for _ in range(n_trials)
    ]
    means.append(np.mean(rates))
    stds.append(np.std(rates))

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].errorbar(episode_counts, means, yerr=stds, fmt='o-', color='steelblue',
                 capsize=4, linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of episodes (log scale)')
axes[0].set_ylabel('Mean win rate')
axes[0].set_title('Win-rate estimate vs episode budget')

axes[1].plot(episode_counts, stds, 's-', color='tomato', linewidth=2)
axes[1].axhline(0.01, color='gray', linestyle='--', label='σ = 0.01 (stable threshold)')
axes[1].set_xscale('log')
axes[1].set_xlabel('Number of episodes (log scale)')
axes[1].set_ylabel('Std dev of win-rate estimate')
axes[1].set_title('Variance shrinks with more episodes')
axes[1].legend()

plt.tight_layout(); plt.show()

print('Episodes | Mean win rate | Std dev')
print('-' * 40)
for n_ep, m, s in zip(episode_counts, means, stds):
    stable = ' ← stable' if s < 0.01 else ''
    print(f'{n_ep:>8d} | {m:.4f}        | {s:.4f}{stable}')

print('\n✅ ~1000 episodes is sufficient for a stable estimate (σ < 0.01).')

### Exercise 3 (Challenge) — Systematic policy for the slippery lake

On the slippery lake each intended action has only a **1/3 probability** of succeeding; with 1/3 probability the agent slips left or right of the intended direction. No hand-crafted deterministic policy can reach 100%.

**Approach**: use **Value Iteration** over the known MDP transition model `env.unwrapped.P` to compute the optimal policy $\pi^*$. This gives the theoretical ceiling (~73%) that no policy can exceed on this stochastic map.

**Key insight about edges**: the optimal policy often moves *away* from the goal to avoid tiles adjacent to holes — counterintuitive but correct given the slip dynamics.

In [ ]:
env_slip = gym.make('FrozenLake-v1', is_slippery=True)
P = env_slip.unwrapped.P
n_states, n_actions, gamma = 16, 4, 0.99

# ── Value Iteration ───────────────────────────────────────────────────────
def value_iteration(P, n_states, n_actions, gamma=0.99, theta=1e-8):
    """Solve for V* and π* via the Bellman optimality equation."""
    V = np.zeros(n_states)
    while True:
        delta = 0
        for s in range(n_states):
            v = V[s]
            V[s] = max(
                sum(prob * (r + gamma * V[ns])
                    for prob, ns, r, done in P[s][a])
                for a in range(n_actions)
            )
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    policy = [
        max(range(n_actions),
            key=lambda a: sum(p * (r + gamma * V[ns]) for p, ns, r, _ in P[s][a]))
        for s in range(n_states)
    ]
    return V, policy

V_star, opt_policy = value_iteration(P, n_states, n_actions)

# ── Evaluate ──────────────────────────────────────────────────────────────
opt_win, _ = evaluate_policy(env_slip, lambda s: opt_policy[s], n_episodes=5000)
orig_win, _ = evaluate_policy(env_slip, lambda s: hand_policy[s], n_episodes=5000)

print(f'Original hand policy on slippery lake:        {orig_win:.2%}')
print(f'Optimal policy (value iteration) win rate:    {opt_win:.2%}')

# ── Visualise the policy on the grid ──────────────────────────────────────
action_symbols = {0: '←', 1: '↓', 2: '→', 3: '↑'}
desc = env_slip.unwrapped.desc
print('\nOptimal policy grid (H=hole, G=goal):')
print('┌' + '───┬' * 3 + '───┐')
for row in range(4):
    cells = []
    for col in range(4):
        s = row * 4 + col
        tile = desc[row][col].decode()
        if tile in ('H', 'G'):
            cells.append(f' {tile} ')
        else:
            cells.append(f' {action_symbols[opt_policy[s]]} ')
    print('│' + '│'.join(cells) + '│')
    if row < 3:
        print('├' + '───┼' * 3 + '───┤')
print('└' + '───┴' * 3 + '───┘')

print('\nV* (optimal state values):')
print(np.round(V_star.reshape(4, 4), 3))
print(f'\n✅ ~73% is the theoretical ceiling — irreducible noise from slip dynamics prevents higher.')